# App Review Scraper
Scrapes reviews from App Store and Play Store across multiple markets.

**Author:** Hugo Malorey  
**Stack:** requests, google-play-scraper, pandas  
**Output schema:** `date | review | rating | source | language | country | answered | username`

> Note: App Store title is concatenated into `review` field for homogeneity.

## 1. Imports

In [1]:
import pandas as pd
import requests
from datetime import datetime, timedelta, timezone
from google_play_scraper import reviews, Sort

## 2. Config

### Appstore Scraping via iTunes RSS feed

The [App Store Connect API](https://developer.apple.com/documentation/appstoreconnectapi/get-v1-apps-_id_-customerreviews) is the official Apple API but requires JWT authentication and access to the target app's App Store Connect account — so it only works if you own the app.

The iTunes RSS feed (`itunes.apple.com/{country}/rss/customerreviews/...`) is an undocumented but long-standing public endpoint that works for **any app** with no auth. Since this scraper is meant to be app-agnostic, the RSS feed is the practical choice. Caveat: it's capped at ~500 reviews per country (10 pages × 50 reviews).

### Playstore Reviews Scraping

For the Play Store, we use the [`google-play-scraper`](https://github.com/JoMingyu/google-play-scraper) Python library, which wraps the same unofficial endpoint the Play Store web UI uses — also no auth required.

## 3. Test connections
Verify both stores are reachable before full scrape.

In [13]:
test_market = MARKETS[0]  # use the first market in the list as a quick sanity check

# ── Test App Store ────────────────────────────────────────────
print("=== App Store connection test ===")
url = (
    f"https://itunes.apple.com/{test_market['country']}/rss/customerreviews"
    f"/page=1/id={APP_STORE_ID}/sortby=mostrecent/json"
)  # Apple's public RSS feed endpoint — returns JSON with the first page of reviews for this app/country
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)

# User-Agent header avoids getting blocked by Apple's servers; timeout prevents hanging indefinitely
entries  = response.json().get("feed", {}).get("entry", [])

# "feed.entry" is where Apple nests the review objects in the RSS JSON response
print(f"Status:         {response.status_code}")
print(f"Reviews found:  {len(entries)} on page 1")

if entries:
    s       = entries[0]  # grab the most recent review
    title   = s.get("title", {}).get("label", "")    # App Store reviews have a separate title field
    content = s.get("content", {}).get("label", "")  # the body text of the review
    review  = " — ".join(filter(None, [title, content]))  # concat title + body, skipping blanks
    print(f"Latest date:    {s.get('updated', {}).get('label', 'N/A')[:10]}")   # slice to YYYY-MM-DD
    print(f"Latest rating:  {s.get('im:rating', {}).get('label', 'N/A')}★")    # rating lives under "im:rating"
    print(f"Sample review:  {review[:100]}...")  # truncate to keep output readable

print()

# ── Test Play Store ───────────────────────────────────────────
print("=== Play Store connection test ===")
result, _ = reviews(
    PLAY_STORE_ID,
    lang=test_market["language"],
    country=test_market["country"],
    sort=Sort.NEWEST,  # sort by newest so we get the most recent review
    count=1            # only fetch 1 review — just enough to confirm the connection works
)
# the second return value is a continuation token for pagination; we don't need it here
print(f"Status:         OK")
print(f"Reviews found:  {len(result)} fetched")
if result:
    s = result[0]
    print(f"Latest date:    {s['at'].strftime('%Y-%m-%d')}")  # 'at' is the review's datetime object
    print(f"Latest rating:  {s['score']}★")                  # 'score' is the star rating (1–5)
    print(f"Sample review:  {str(s['content'])[:100]}...")    # 'content' is the review body text

=== App Store connection test ===
Status:         200
Reviews found:  50 on page 1
Latest date:    2026-04-06
Latest rating:  4★
Sample review:  Problème mise à jours — Bonjour depuis la dernière mise à jours je n’ai plus accès à l’application u...

=== Play Store connection test ===
Status:         OK
Reviews found:  1 fetched
Latest date:    2026-04-08
Latest rating:  1★
Sample review:  Impossible d'ouvrir l'application depuis la dernière mise à jour...


## 4. Scrape App Store (iOS)

In [14]:
def scrape_app_store(app_store_id, markets, cutoff_date, min_rating, max_rating, pages=5):
    """Scrape App Store reviews. Title concatenated into review field."""
    results = []
    for market in markets:
        country, language = market["country"], market["language"]
        print(f"  [{country}] scraping...")
        count = 0
        for page in range(1, pages + 1):
            url = (
                f"https://itunes.apple.com/{country}/rss/customerreviews"
                f"/page={page}/id={app_store_id}/sortby=mostrecent/json"
            )
            try:
                resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            except requests.exceptions.Timeout:
                print(f"  [{country}] page {page} timeout — stopping")
                break

            if resp.status_code != 200:
                break

            entries = resp.json().get("feed", {}).get("entry", [])
            if not entries:
                break

            # Apple returns a dict instead of list when only 1 entry — normalize
            if isinstance(entries, dict):
                entries = [entries]

            stop = False
            for entry in entries:
                rating   = int(entry.get("im:rating", {}).get("label", 5))
                date_str = entry.get("updated", {}).get("label", "")[:10]
                date     = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)

                if date < cutoff_date:
                    stop = True
                    break

                if not (min_rating <= rating <= max_rating):
                    continue

                title   = entry.get("title", {}).get("label", "")
                content = entry.get("content", {}).get("label", "")
                review  = " — ".join(filter(None, [title, content]))

                results.append({
                    "date":     date,
                    "review":   review,
                    "rating":   rating,
                    "source":   "App Store",
                    "language": language,
                    "country":  country,
                    "answered": False,
                    "username": entry.get("author", {}).get("name", {}).get("label", "anonymous"),
                })
                count += 1

            if stop:
                print(f"  [{country}] cutoff reached at page {page} — stopping")
                break

        print(f"  [{country}] → {count} reviews")
    return results


print("Scraping App Store...")
ios_reviews = scrape_app_store(APP_STORE_ID, MARKETS, CUTOFF_DATE, MIN_RATING, MAX_RATING)
print(f"Total iOS: {len(ios_reviews)}")

Scraping App Store...
  [fr] scraping...
  [fr] cutoff reached at page 2 — stopping
  [fr] → 30 reviews
  [de] scraping...
  [de] cutoff reached at page 1 — stopping
  [de] → 3 reviews
  [gb] scraping...
  [gb] → 0 reviews
  [es] scraping...
  [es] cutoff reached at page 1 — stopping
  [es] → 0 reviews
  [it] scraping...
  [it] cutoff reached at page 1 — stopping
  [it] → 1 reviews
  [nl] scraping...
  [nl] cutoff reached at page 1 — stopping
  [nl] → 0 reviews
  [be] scraping...
  [be] cutoff reached at page 1 — stopping
  [be] → 2 reviews
  [pt] scraping...
  [pt] cutoff reached at page 1 — stopping
  [pt] → 0 reviews
  [at] scraping...
  [at] → 0 reviews
Total iOS: 36


## 5. Scrape Play Store (Android)

In [15]:
def scrape_play_store(play_store_id, markets, cutoff_date, min_rating, max_rating, count=300):
    """Scrape Play Store reviews. No title field on this platform."""
    results = []
    for market in markets:
        country, language = market["country"], market["language"]
        print(f"  [{country}] scraping...")
        market_count = 0
        for star in range(min_rating, max_rating + 1):
            fetched, _ = reviews(
                play_store_id,
                lang=language,
                country=country,
                sort=Sort.NEWEST,
                count=count,
                filter_score_with=star,
            )
            star_count = 0
            for r in fetched:
                date = r["at"]
                if date.tzinfo is None:
                    date = date.replace(tzinfo=timezone.utc)
                if date < cutoff_date:
                    continue
                results.append({
                    "date":     date,
                    "review":   r.get("content", ""),
                    "rating":   r["score"],
                    "source":   "Play Store",
                    "language": language,
                    "country":  country,
                    "answered": r.get("replyContent") is not None,
                    "username": r.get("userName", "anonymous"),
                })
                star_count   += 1
                market_count += 1
            print(f"  [{country}] {star}★: {star_count} reviews")
        print(f"  [{country}] → {market_count} total")
    return results


print("Scraping Play Store...")
android_reviews = scrape_play_store(PLAY_STORE_ID, MARKETS, CUTOFF_DATE, MIN_RATING, MAX_RATING)
print(f"Total Android: {len(android_reviews)}")

Scraping Play Store...
  [fr] scraping...
  [fr] 1★: 50 reviews
  [fr] 2★: 14 reviews
  [fr] → 64 total
  [de] scraping...
  [de] 1★: 1 reviews
  [de] 2★: 0 reviews
  [de] → 1 total
  [gb] scraping...
  [gb] 1★: 1 reviews
  [gb] 2★: 0 reviews
  [gb] → 1 total
  [es] scraping...
  [es] 1★: 3 reviews
  [es] 2★: 0 reviews
  [es] → 3 total
  [it] scraping...
  [it] 1★: 1 reviews
  [it] 2★: 0 reviews
  [it] → 1 total
  [nl] scraping...
  [nl] 1★: 0 reviews
  [nl] 2★: 0 reviews
  [nl] → 0 total
  [be] scraping...
  [be] 1★: 50 reviews
  [be] 2★: 14 reviews
  [be] → 64 total
  [pt] scraping...
  [pt] 1★: 0 reviews
  [pt] 2★: 2 reviews
  [pt] → 2 total
  [at] scraping...
  [at] 1★: 1 reviews
  [at] 2★: 0 reviews
  [at] → 1 total
Total Android: 137


## 6. Merge & clean

In [17]:
df = pd.DataFrame(ios_reviews + android_reviews, columns=COLUMNS)
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date", ascending=False).reset_index(drop=True)

print(f"{'='*40}")
print(f"TOTAL:           {len(df)} reviews")
print(f"{'='*40}")
print(f"App Store:       {len(df[df['source'] == 'App Store'])}")
print(f"Play Store:      {len(df[df['source'] == 'Play Store'])}")
print(f"Answered:        {df['answered'].sum()} ({df['answered'].mean()*100:.0f}%)")
print()
print("Rating distribution:")
print(df['rating'].value_counts().sort_index().to_string())
print()
df.head(20)

TOTAL:           173 reviews
App Store:       36
Play Store:      137
Answered:        121 (70%)

Rating distribution:
rating
1    132
2     41



,date,review,rating,source,language,country,answered,username
0,2026-04-08 04:49:40+00:00,Impossible d'ouvrir l'application depuis la de...,1,Play Store,fr,fr,True,THT PVS
1,2026-04-08 04:49:40+00:00,Impossible d'ouvrir l'application depuis la de...,1,Play Store,fr,be,True,THT PVS
2,2026-04-06 11:24:58+00:00,la synchronisation est une catastrophe! Pas au...,2,Play Store,fr,fr,False,Lidvine Charpentier
3,2026-04-06 11:24:58+00:00,la synchronisation est une catastrophe! Pas au...,2,Play Store,fr,be,False,Lidvine Charpentier
4,2026-04-05 13:26:47+00:00,affichage du portefeuille divergent entre plus...,1,Play Store,fr,be,True,Jm Cds
5,2026-04-05 13:26:47+00:00,affichage du portefeuille divergent entre plus...,1,Play Store,fr,fr,True,Jm Cds
6,2026-04-05 00:00:00+00:00,Nul — J’ai même pas pu essayer l’appli que mon...,1,App Store,fr,fr,False,Ynaftis
7,2026-04-04 15:37:55+00:00,mis de l'argent sur l'application et depuis el...,1,Play Store,fr,fr,False,Florian Pontzeele
8,2026-04-04 15:37:55+00:00,mis de l'argent sur l'application et depuis el...,1,Play Store,fr,be,False,Florian Pontzeele
9,2026-04-02 10:07:58+00:00,Llevo varios meses con la ella instalada y no ...,1,Play Store,es,es,False,Francisco Centoira Dopazo


## 7. Print all reviews (readable)

In [18]:
for i, row in df.iterrows():
    print(f"{'─'*60}")
    print(f"#{i+1} [{row['source']} | {row['rating']}★ | {row['country'].upper()} | {row['date'].strftime('%Y-%m-%d')} | answered: {row['answered']}]")
    print(f"Review:  {row['review']}")
    print(f"User:    {row['username']}")
print(f"{'─'*60}")
print(f"TOTAL: {len(df)} reviews")

────────────────────────────────────────────────────────────
#1 [Play Store | 1★ | FR | 2026-04-08 | answered: True]
Review:  Impossible d'ouvrir l'application depuis la dernière mise à jour
User:    THT PVS
────────────────────────────────────────────────────────────
#2 [Play Store | 1★ | BE | 2026-04-08 | answered: True]
Review:  Impossible d'ouvrir l'application depuis la dernière mise à jour
User:    THT PVS
────────────────────────────────────────────────────────────
#3 [Play Store | 2★ | FR | 2026-04-06 | answered: False]
Review:  la synchronisation est une catastrophe! Pas automatique, les transactions apparaissent disparaissent...
User:    Lidvine Charpentier
────────────────────────────────────────────────────────────
#4 [Play Store | 2★ | BE | 2026-04-06 | answered: False]
Review:  la synchronisation est une catastrophe! Pas automatique, les transactions apparaissent disparaissent...
User:    Lidvine Charpentier
────────────────────────────────────────────────────────────
#5 

## 8. Export to CSV

In [9]:
output_file = f"{APP_NAME}_reviews_{datetime.now().strftime('%Y%m%d')}_{DAYS_BACK}d.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved → {output_file}")

Saved → bitstack_reviews_20260409_90d.csv


## 9. Format for Claude analysis

In [19]:
lines = []
for _, row in df.iterrows():
    answered_part = " [answered]" if row['answered'] else ""
    lines.append(
        f"[{row['source']} | {row['rating']}★ | {row['country'].upper()} | "
        f"{row['date'].strftime('%Y-%m-%d')}{answered_part}]\n"
        f"{row['review']}\n"
    )

claude_input = "\n".join(lines)

print(f"Total reviews:    {len(df)}")
print(f"Total characters: {len(claude_input)}")
print()
print("--- SAMPLE (first 3) ---")
print("\n".join(lines[:3]))

Total reviews:    173
Total characters: 45535

--- SAMPLE (first 3) ---
[Play Store | 1★ | FR | 2026-04-08 [answered]]
Impossible d'ouvrir l'application depuis la dernière mise à jour

[Play Store | 1★ | BE | 2026-04-08 [answered]]
Impossible d'ouvrir l'application depuis la dernière mise à jour

[Play Store | 2★ | FR | 2026-04-06]
la synchronisation est une catastrophe! Pas automatique, les transactions apparaissent disparaissent...

